In [1]:
import os
import sys
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [2]:
# dataset/LA_train/LA_train.zip
!cp /content/drive/MyDrive/lvths/dataset/LA_train/LA_train.zip /content/
!unzip -q /content/LA_train.zip -d /content/LA_train_data/
# -q là chế độ quiet, giúp Colab không in ra hàng chục ngàn dòng giải nén gây đơ trình duyệt

In [3]:
# Cập nhật đường dẫn tương ứng với thư mục bạn vừa giải nén
PROTOCOL_PATH = "/content/LA_train_data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
AUDIO_DIR = "/content/LA_train_data/LA/ASVspoof2019_LA_train/flac/"

In [4]:
PROJECT_PATH = "/content/drive/MyDrive/lvths/aasist_baseline/"
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)

from models.AASIST import Model as RealAASIST

In [5]:
# ==========================================
# 2. ĐỊNH NGHĨA DATASET
# ==========================================
class ASVspoof2019LA(Dataset):
    def __init__(self, protocol_file, audio_dir, max_frames=64000, target_sr=16000):
        self.audio_dir = audio_dir
        self.max_frames = max_frames
        self.target_sr = target_sr
        self.data_list = []
        with open(protocol_file, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    label = 0 if parts[4] == 'bonafide' else 1
                    self.data_list.append((parts[1], label))

    def __len__(self): return len(self.data_list)

    def process_audio(self, waveform, sr):
        if sr != self.target_sr:
            waveform = torchaudio.transforms.Resample(sr, self.target_sr)(waveform)
        num_frames = waveform.shape[1]
        if num_frames > self.max_frames:
            waveform = waveform[:, :self.max_frames]
        elif num_frames < self.max_frames:
            waveform = F.pad(waveform, (0, self.max_frames - num_frames))
        return waveform

    def __getitem__(self, idx):
        audio_id, label = self.data_list[idx]
        file_path = os.path.join(self.audio_dir, f"{audio_id}.flac")
        waveform, sr = torchaudio.load(file_path)
        if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
        waveform = self.process_audio(waveform, sr).squeeze(0)
        return waveform, torch.tensor(label, dtype=torch.long)


In [6]:
# ==========================================
# 3. THUẬT TOÁN TẤN CÔNG FGSM (FLOAT32)
# ==========================================
def fgsm_attack(model, criterion, waveforms, labels, epsilon=0.005):
    """Sinh nhiễu FGSM bằng Float32 nguyên bản để tránh NaN"""
    waveforms_adv = waveforms.clone().detach().requires_grad_(True)

    # Model AASIST của bạn trả về tuple (logits, features), ta chỉ lấy logits
    logits, _ = model(waveforms_adv)
    loss = criterion(logits, labels)

    model.zero_grad()
    loss.backward()
    perturbed = waveforms_adv + epsilon * waveforms_adv.grad.data.sign()
    return torch.clamp(perturbed, -1.0, 1.0).detach()

In [7]:
# ==========================================
# 4. KHỞI TẠO VÀ CHẠY HUẤN LUYỆN (BASELINE)
# ==========================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Khởi động Huấn luyện BASELINE trên: {device}")

    # --- CẤU HÌNH ĐƯỜNG DẪN ---
    # PROTOCOL_PATH = "/content/LA_train_data/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
    # AUDIO_DIR = "/content/LA_train_data/ASVspoof2019_LA_train/flac/"

    # Dùng batch_size 24 để tránh OOM do không dùng AMP (Float16)
    train_dataset = ASVspoof2019LA(PROTOCOL_PATH, AUDIO_DIR)
    train_loader = DataLoader(train_dataset, batch_size=24, shuffle=True, num_workers=8, pin_memory=True)

    # --- KHỞI TẠO MÔ HÌNH ---
    CONFIG_PATH = os.path.join(PROJECT_PATH, "config/AASIST.conf")
    with open(CONFIG_PATH, "r") as f:
        model_config = json.load(f)["model_config"]

    model_baseline = RealAASIST(model_config).to(device)
    print("✅ Đã khởi tạo Mô hình Baseline AASIST.")

    # --- LOSS & OPTIMIZER ---
    task_criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model_baseline.parameters(), lr=1e-4)

    # --- SIÊU THAM SỐ VÀ CHECKPOINT ---
    epochs = 20
    epsilon = 0.002
    best_loss = float('inf')
    best_epoch = -1
    SAVE_PATH = "/content/drive/MyDrive/baseline_aasist_fgsm_best.pth"

    # --- VÒNG LẶP HUẤN LUYỆN ---
    for epoch in range(epochs):
        model_baseline.train()
        running_task_loss = 0.0
        valid_batches = 0

        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for waveforms, labels in progress_bar:
            waveforms, labels = waveforms.to(device), labels.to(device)

            # 1. Sinh nhiễu FGSM
            model_baseline.eval()
            noisy_waveforms = fgsm_attack(model_baseline, task_criterion, waveforms, labels, epsilon)
            model_baseline.train()

            # 2. Huấn luyện trực tiếp trên bản nhiễu (Không có Teacher)
            optimizer.zero_grad()
            logits, _ = model_baseline(noisy_waveforms)
            loss_task = task_criterion(logits, labels)

            # Bỏ qua nếu xuất hiện NaN đột xuất
            if torch.isnan(loss_task):
                continue

            loss_task.backward()

            # Cắt xén đạo hàm (Gradient Clipping)
            torch.nn.utils.clip_grad_norm_(model_baseline.parameters(), max_norm=5.0)

            optimizer.step()

            running_task_loss += loss_task.item()
            valid_batches += 1

            progress_bar.set_postfix({'TaskLoss': f'{running_task_loss/valid_batches:.4f}'})

        # --- LƯU BEST MODEL ---
        if valid_batches > 0:
            avg_epoch_loss = running_task_loss / valid_batches
            print(f"\n=> Kết thúc Epoch {epoch+1} | Avg Task Loss: {avg_epoch_loss:.4f}")

            if avg_epoch_loss < best_loss:
                best_loss = avg_epoch_loss
                best_epoch = epoch + 1
                torch.save(model_baseline.state_dict(), SAVE_PATH)
                print(f"🌟 KỶ LỤC MỚI! Đã lưu Baseline Model tại Epoch {best_epoch} (Loss: {best_loss:.4f})\n")
            else:
                print(f"📉 Loss không giảm. Baseline Model vẫn ở Epoch {best_epoch} (Loss: {best_loss:.4f})\n")

    print(f"🎉 Hoàn tất huấn luyện Baseline! Trọng số tốt nhất được lưu tại: {SAVE_PATH}")

🚀 Khởi động Huấn luyện BASELINE trên: cuda
✅ Đã khởi tạo Mô hình Baseline AASIST.


Epoch 1/20: 100%|██████████| 1058/1058 [08:21<00:00,  2.11it/s, TaskLoss=3.9499]



=> Kết thúc Epoch 1 | Avg Task Loss: 3.9499
🌟 KỶ LỤC MỚI! Đã lưu Baseline Model tại Epoch 1 (Loss: 3.9499)



Epoch 2/20: 100%|██████████| 1058/1058 [08:19<00:00,  2.12it/s, TaskLoss=3.0694]



=> Kết thúc Epoch 2 | Avg Task Loss: 3.0694
🌟 KỶ LỤC MỚI! Đã lưu Baseline Model tại Epoch 2 (Loss: 3.0694)



...


=> Kết thúc Epoch 18 | Avg Task Loss: 2.6407
📉 Loss không giảm. Baseline Model vẫn ở Epoch 15 (Loss: 2.6362)



Epoch 19/20: 100%|██████████| 1058/1058 [08:19<00:00,  2.12it/s, TaskLoss=2.6405]



=> Kết thúc Epoch 19 | Avg Task Loss: 2.6405
📉 Loss không giảm. Baseline Model vẫn ở Epoch 15 (Loss: 2.6362)



Epoch 20/20: 100%|██████████| 1058/1058 [08:19<00:00,  2.12it/s, TaskLoss=2.6130]


=> Kết thúc Epoch 20 | Avg Task Loss: 2.6130
🌟 KỶ LỤC MỚI! Đã lưu Baseline Model tại Epoch 20 (Loss: 2.6130)

🎉 Hoàn tất huấn luyện Baseline! Trọng số tốt nhất được lưu tại: /content/drive/MyDrive/baseline_aasist_fgsm_best.pth
